# NBA Next-Game PRA Prediction
### Leakage-safe player forecasting with player history, team context, Bayesian optimization, ensembling, and uncertainty

**Target:** predict a player's **Points + Rebounds + Assists (PRA)** for the next game in which the player appears.

This notebook uses four supplied datasets:
- `PlayerStatistics(2).csv`
- `Players(1).csv`
- `TeamStatistics(2).csv`
- `TeamHistories(2).csv`

## 1. Project goal
Build a portfolio-grade forecasting pipeline that answers a practical question: **given everything known before tip-off, what PRA should we expect from a player in the next game?**

The core design choice is temporal discipline. Every rolling player/team statistic is shifted so the current game's box score cannot leak into its own prediction.

## 2. Modeling strategy
1. Clean and audit all four datasets.
2. Construct pregame player form from prior games only.
3. Construct pregame team and opponent form from prior team games only.
4. Add static player profile information.
5. Split chronologically into train / validation / test periods.
6. Compare a rolling baseline and several distinct regression models.
7. Tune a gradient-boosted model with Optuna Bayesian optimization.
8. Blend strong models using validation-set nonnegative least squares.
9. Calibrate an 80% residual interval.
10. Refit production models and expose `predict_next_pra(...)`.

In [ ]:
# Sections 1-2 are explanatory; no executable code in this block.

## 3. Leakage rule
For a game on date **t**, feature values may use information from dates **< t** plus known pregame context (home/away, opponent, competition type, player bio). They may **not** use minutes, points, rebounds, assists, shooting, or team results from date **t**.

This is why rolling features below use `.shift(1)` before the rolling calculation.

## 4. Configuration
The default modeling era begins in 2010, while feature history begins in 2008 so early modeling rows still have useful lag context. This keeps the notebook relevant to the modern NBA and practical to run locally.

In [ ]:
from pathlib import Path
import warnings, json, math
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

from scipy.optimize import nnls

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)
RANDOM_STATE = 42

# ---- next section ----

DATA_DIR = Path('.')

PLAYER_STATS_PATH = DATA_DIR / 'PlayerStatistics(2).csv'
PLAYERS_PATH = DATA_DIR / 'Players(1).csv'
TEAM_STATS_PATH = DATA_DIR / 'TeamStatistics(2).csv'
TEAM_HISTORY_PATH = DATA_DIR / 'TeamHistories(2).csv'

# If running this downloaded notebook from another folder, edit DATA_DIR above.
FEATURE_START_DATE = pd.Timestamp('2013-01-01')
MODEL_START_DATE = pd.Timestamp('2015-01-01')
VALID_START_DATE = pd.Timestamp('2023-07-01')
TEST_START_DATE = pd.Timestamp('2024-07-01')

MIN_MINUTES_PLAYED = 0.0
MAX_TRAIN_ROWS = 180_000
TUNE_MAX_ROWS = 60_000
RUN_OPTUNA = True
OPTUNA_TRIALS = 20
INTERVAL_COVERAGE = 0.80

VALID_GAME_TYPES = {
    'Regular Season', 'Playoffs', 'Play-in Tournament',
    'NBA Emirates Cup', 'Emirates NBA Cup', 'NBA Cup', 'in-season-knockout'
}

## 5. Load only the columns needed from the large player-game file
The raw player statistics file is much larger than the other inputs, so loading a focused column set reduces memory pressure without sacrificing useful modeling information.

## 6. Raw dataset sizes
These checks establish the unit of observation before any filtering or joining.

In [ ]:
PLAYER_STAT_COLS = [
    'firstName','lastName','personId','gameId','gameDateTimeEst',
    'playerteamCity','playerteamName','opponentteamCity','opponentteamName',
    'gameType','win','home','numMinutes','points','assists','blocks','steals',
    'fieldGoalsAttempted','fieldGoalsMade','fieldGoalsPercentage',
    'threePointersAttempted','threePointersMade','threePointersPercentage',
    'freeThrowsAttempted','freeThrowsMade','freeThrowsPercentage',
    'reboundsDefensive','reboundsOffensive','reboundsTotal','foulsPersonal',
    'turnovers','plusMinusPoints','playerteamId','opponentteamId',
    'comment','startingPosition','gameDate'
]

player_stats = pd.read_csv(PLAYER_STATS_PATH, usecols=PLAYER_STAT_COLS, low_memory=False)
players = pd.read_csv(PLAYERS_PATH, low_memory=False)
team_stats = pd.read_csv(TEAM_STATS_PATH, low_memory=False)
team_history = pd.read_csv(TEAM_HISTORY_PATH, low_memory=False)

# ---- next section ----

raw_shapes = pd.DataFrame({
    'dataset': ['player_stats','players','team_stats','team_history'],
    'rows': [len(player_stats), len(players), len(team_stats), len(team_history)],
    'columns': [player_stats.shape[1], players.shape[1], team_stats.shape[1], team_history.shape[1]]
})
raw_shapes

## 7. Schema snapshots
A quick schema review helps identify mixed-type fields such as minutes and historical identifiers.

## 8. Standardize dates and numeric columns
Bad parses are converted to missing values rather than causing silent string comparisons later.

In [ ]:
for name, df in {
    'player_stats': player_stats,
    'players': players,
    'team_stats': team_stats,
    'team_history': team_history
}.items():
    print(f'\n{name}:')
    print(df.dtypes.astype(str).to_string())

# ---- next section ----

player_stats['gameDateTimeEst'] = pd.to_datetime(player_stats['gameDateTimeEst'], errors='coerce')
player_stats['gameDate'] = pd.to_datetime(player_stats['gameDate'], errors='coerce')
team_stats['gameDateTimeEst'] = pd.to_datetime(team_stats['gameDateTimeEst'], errors='coerce')
team_stats['gameDate'] = pd.to_datetime(team_stats['gameDate'], errors='coerce')
players['birthDate'] = pd.to_datetime(players['birthDate'], errors='coerce')

PLAYER_NUMERIC = [
    'personId','home','numMinutes','points','assists','blocks','steals',
    'fieldGoalsAttempted','fieldGoalsMade','fieldGoalsPercentage',
    'threePointersAttempted','threePointersMade','threePointersPercentage',
    'freeThrowsAttempted','freeThrowsMade','freeThrowsPercentage',
    'reboundsDefensive','reboundsOffensive','reboundsTotal','foulsPersonal',
    'turnovers','plusMinusPoints','playerteamId','opponentteamId'
]
for col in PLAYER_NUMERIC:
    player_stats[col] = pd.to_numeric(player_stats[col], errors='coerce')

## 9. Missingness audit
Missingness is expected in early historical data and in player metadata. The model pipeline later imputes predictors; the target components themselves must be present.

## 10. Duplicate-key audit
A player-game should be unique on `(personId, gameId)`, and a team-game should be unique on `(teamId, gameId)`.

In [ ]:
missing_player = (player_stats.isna().mean().sort_values(ascending=False) * 100).round(2)
missing_player.head(20).to_frame('missing_%')

# ---- next section ----

print('Duplicate player-game rows:', player_stats.duplicated(['personId','gameId']).sum())
print('Duplicate team-game rows:', team_stats.duplicated(['teamId','gameId']).sum())
print('Duplicate player metadata IDs:', players.duplicated(['personId']).sum())

## 11. Historical coverage
Long historical coverage is valuable for audit purposes, but the actual model focuses on a modern era where roles, pace, and shot profiles are more comparable.

## 12. Competition types
Preseason and All-Star events are excluded from the primary model because rotations and incentives are materially different from normal competitive games.

In [ ]:
coverage = pd.Series({
    'player_games_min': player_stats['gameDateTimeEst'].min(),
    'player_games_max': player_stats['gameDateTimeEst'].max(),
    'team_games_min': team_stats['gameDateTimeEst'].min(),
    'team_games_max': team_stats['gameDateTimeEst'].max(),
})
coverage

# ---- next section ----

player_stats['gameType'].value_counts(dropna=False).head(15)

## 13. Prepare player metadata
Static player attributes are useful priors for role and production, especially when a player has relatively little NBA history.

## 14. Prepare a current team lookup
`TeamHistories` is used for stable abbreviations and readable outputs. Historical franchise names remain available in the raw game rows.

In [ ]:
player_bio = players[[
    'personId','firstName','lastName','birthDate','heightInches','bodyWeightLbs',
    'guard','forward','center','draftYear','draftRound','draftNumber','fromYear','toYear'
]].copy()

for col in ['heightInches','bodyWeightLbs','guard','forward','center','draftYear','draftRound','draftNumber','fromYear','toYear']:
    player_bio[col] = pd.to_numeric(player_bio[col], errors='coerce')

player_bio = player_bio.drop_duplicates('personId')
player_bio.head()

# ---- next section ----

team_lookup = (
    team_history.sort_values(['teamId','seasonActiveTill'])
    .groupby('teamId', as_index=False)
    .tail(1)[['teamId','teamCity','teamName','teamAbbrev','league']]
    .rename(columns={'teamId':'lookupTeamId'})
)
team_lookup.head(35)

## 15. Define the PRA target
PRA is intentionally transparent: `points + reboundsTotal + assists`. It is calculated only for games with valid box-score components.

## 16. Filter to usable competitive appearances
Rows with no valid player ID, game date, PRA, or positive appearance time cannot teach the model about on-court production.

In [ ]:
player_stats['PRA'] = player_stats['points'] + player_stats['reboundsTotal'] + player_stats['assists']
player_stats[['points','reboundsTotal','assists','PRA']].describe().round(2)

# ---- next section ----

pg = player_stats[
    player_stats['personId'].notna()
    & player_stats['gameDateTimeEst'].notna()
    & player_stats['PRA'].notna()
    & player_stats['gameType'].isin(VALID_GAME_TYPES)
    & (player_stats['gameDateTimeEst'] >= FEATURE_START_DATE)
    & (player_stats['numMinutes'] > MIN_MINUTES_PLAYED)
].copy()

pg['personId'] = pg['personId'].astype('int64')
pg['playerteamId'] = pg['playerteamId'].astype('Int64')
pg['opponentteamId'] = pg['opponentteamId'].astype('Int64')
pg = pg.sort_values(['personId','gameDateTimeEst','gameId']).reset_index(drop=True)
print(f'Usable player appearances: {len(pg):,}')
print(f'Players: {pg.personId.nunique():,}')

## 17. Create readable player names
The ID remains the modeling key; names are display fields only.

## 18. Previous-game rest and experience
`rest_days` and `career_games_before` are both known before the current game. Rest is capped to reduce the influence of offseasons and long injury absences.

In [ ]:
pg['playerName'] = (
    pg['firstName'].fillna('').str.strip() + ' ' + pg['lastName'].fillna('').str.strip()
).str.strip()
pg[['personId','playerName']].drop_duplicates().head()

# ---- next section ----

g = pg.groupby('personId', sort=False)
pg['previous_game_date'] = g['gameDateTimeEst'].shift(1)
pg['rest_days'] = (pg['gameDateTimeEst'] - pg['previous_game_date']).dt.total_seconds() / 86400
pg['rest_days'] = pg['rest_days'].clip(lower=0, upper=14)
pg['career_games_before'] = g.cumcount()

## 19. Lag-one player features
The immediately preceding game captures short-term role, workload, and production shifts.

## 20. Rolling player-form windows
Multiple windows let the models distinguish hot/cold short-term form from a more stable role baseline.

In [ ]:
PLAYER_LAG_COLS = [
    'PRA','points','reboundsTotal','assists','numMinutes','fieldGoalsAttempted',
    'threePointersAttempted','freeThrowsAttempted','turnovers','fieldGoalsPercentage',
    'threePointersPercentage','freeThrowsPercentage','plusMinusPoints'
]
for col in PLAYER_LAG_COLS:
    pg[f'lag1_{col}'] = g[col].shift(1)

# ---- next section ----

ROLL_WINDOWS = [3, 5, 10, 20]
ROLL_COLS = ['PRA','points','reboundsTotal','assists','numMinutes']

for col in ROLL_COLS:
    shifted = g[col].shift(1)
    for w in ROLL_WINDOWS:
        pg[f'{col}_mean_{w}'] = (
            shifted.groupby(pg['personId']).rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
        )
        pg[f'{col}_std_{w}'] = (
            shifted.groupby(pg['personId']).rolling(w, min_periods=2).std().reset_index(level=0, drop=True)
        )

## 21. Rolling usage and efficiency context
Attempts and turnovers help represent offensive role independently of whether shots happened to fall in the previous game.

## 22. Prior starting-rate feature
Current-game starting position would not always be known at forecast time, so the model instead uses the player's prior 10-game starting rate.

In [ ]:
USAGE_COLS = ['fieldGoalsAttempted','threePointersAttempted','freeThrowsAttempted','turnovers']
for col in USAGE_COLS:
    shifted = g[col].shift(1)
    for w in [5, 10]:
        pg[f'{col}_mean_{w}'] = (
            shifted.groupby(pg['personId']).rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
        )

# ---- next section ----

pg['was_starter'] = pg['startingPosition'].notna().astype(float)
starter_shift = pg.groupby('personId')['was_starter'].shift(1)
pg['starter_rate_10'] = (
    starter_shift.groupby(pg['personId']).rolling(10, min_periods=1).mean().reset_index(level=0, drop=True)
)

## 23. Long-run player baseline
An expanding pregame mean gives the models a stable career-level anchor without using the current result.

## 24. Clean team-game statistics
Team context is built independently from the player table and then merged by game ID and team ID.

In [ ]:
shifted_pra = pg.groupby('personId')['PRA'].shift(1)
pg['career_pra_mean'] = (
    shifted_pra.groupby(pg['personId']).expanding(min_periods=1).mean().reset_index(level=0, drop=True)
)

# ---- next section ----

ts = team_stats.copy()
ts = ts[ts['gameDateTimeEst'].notna() & (ts['gameDateTimeEst'] >= FEATURE_START_DATE)].copy()

TEAM_NUMERIC = [
    'teamId','opponentTeamId','teamScore','opponentScore','assists','fieldGoalsAttempted',
    'fieldGoalsPercentage','threePointersAttempted','freeThrowsAttempted','reboundsTotal',
    'turnovers','plusMinusPoints','home'
]
for col in TEAM_NUMERIC:
    ts[col] = pd.to_numeric(ts[col], errors='coerce')

ts = ts.sort_values(['teamId','gameDateTimeEst','gameId']).reset_index(drop=True)
print(f'Team-game rows used for context: {len(ts):,}')

## 25. Pregame team form
Again, every team rolling metric is shifted. The value attached to a current game reflects only that team's earlier games.

## 26. Team rest
Team-level rest can differ from a player's personal rest after absences, so both signals are retained.

In [ ]:
TEAM_FORM_BASE = [
    'teamScore','opponentScore','assists','fieldGoalsAttempted','fieldGoalsPercentage',
    'threePointersAttempted','freeThrowsAttempted','reboundsTotal','turnovers','plusMinusPoints'
]

tg = ts.groupby('teamId', sort=False)
for col in TEAM_FORM_BASE:
    shifted = tg[col].shift(1)
    ts[f'team_{col}_mean_10'] = (
        shifted.groupby(ts['teamId']).rolling(10, min_periods=1).mean().reset_index(level=0, drop=True)
    )

# ---- next section ----

ts['team_previous_date'] = tg['gameDateTimeEst'].shift(1)
ts['team_rest_days'] = ((ts['gameDateTimeEst'] - ts['team_previous_date']).dt.total_seconds() / 86400).clip(0, 14)

## 27. Merge own-team pregame context
This adds recent offense, rebounding, ball movement, and workload environment for the player's team.

## 28. Merge opponent pregame context
The opponent's shifted `opponentScore` rolling mean is especially useful because it approximates recent points allowed before the matchup.

In [ ]:
TEAM_FORM_COLS = [c for c in ts.columns if c.startswith('team_') and c.endswith('_mean_10')] + ['team_rest_days']
own_ctx = ts[['gameId','teamId'] + TEAM_FORM_COLS].copy()
own_ctx = own_ctx.rename(columns={'teamId':'playerteamId'})
pg = pg.merge(own_ctx, on=['gameId','playerteamId'], how='left', validate='many_to_one')

# ---- next section ----

opp_ctx = ts[['gameId','teamId'] + TEAM_FORM_COLS].copy()
opp_ctx = opp_ctx.rename(columns={
    'teamId':'opponentteamId',
    **{c:f'opp_{c}' for c in TEAM_FORM_COLS}
})
pg = pg.merge(opp_ctx, on=['gameId','opponentteamId'], how='left', validate='many_to_one')

## 29. Merge player profile features
Age is calculated at the game date so it changes appropriately across a player's career.

## 30. Season-phase features
Month and day-of-season are low-complexity context signals for changing rotations and fatigue patterns.

In [ ]:
BIO_COLS = ['personId','birthDate','heightInches','bodyWeightLbs','guard','forward','center','draftNumber']
pg = pg.merge(player_bio[BIO_COLS], on='personId', how='left', validate='many_to_one')
pg['age_years'] = (pg['gameDateTimeEst'] - pg['birthDate']).dt.total_seconds() / (365.25 * 86400)

# ---- next section ----

pg['game_month'] = pg['gameDateTimeEst'].dt.month
season_year = np.where(pg['game_month'] >= 7, pg['gameDateTimeEst'].dt.year, pg['gameDateTimeEst'].dt.year - 1)
pg['season_start'] = pd.to_datetime(pd.Series(season_year, index=pg.index).astype(str) + '-07-01')
pg['days_into_season'] = (pg['gameDateTimeEst'].dt.normalize() - pg['season_start']).dt.days

## 31. Pregame feature list
No current-game box-score variable is included. The target `PRA` is retained separately only for supervised learning.

## 32. Explicit leakage audit
This guardrail fails loudly if any same-game production metric accidentally appears in the model feature set without a lag/rolling prefix.

In [ ]:
NUMERIC_FEATURES = [
    'home','rest_days','career_games_before','career_pra_mean','starter_rate_10','age_years',
    'heightInches','bodyWeightLbs','guard','forward','center','draftNumber','game_month','days_into_season',
    *[f'lag1_{c}' for c in PLAYER_LAG_COLS],
    *[f'{c}_{stat}_{w}' for c in ROLL_COLS for stat in ['mean','std'] for w in ROLL_WINDOWS],
    *[f'{c}_mean_{w}' for c in USAGE_COLS for w in [5,10]],
    *TEAM_FORM_COLS,
    *[f'opp_{c}' for c in TEAM_FORM_COLS],
]
CATEGORICAL_FEATURES = ['playerteamId','opponentteamId','gameType']
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

# preserve order while removing accidental duplicates
FEATURES = list(dict.fromkeys(FEATURES))
NUMERIC_FEATURES = [c for c in FEATURES if c not in CATEGORICAL_FEATURES]
print('Total model features:', len(FEATURES))

# ---- next section ----

FORBIDDEN_CURRENT_GAME = {
    'PRA','points','reboundsTotal','assists','numMinutes','fieldGoalsAttempted','fieldGoalsMade',
    'threePointersAttempted','threePointersMade','freeThrowsAttempted','freeThrowsMade','turnovers',
    'plusMinusPoints','startingPosition','win'
}
leaks = sorted(set(FEATURES) & FORBIDDEN_CURRENT_GAME)
assert not leaks, f'Leakage features found: {leaks}'
print('Leakage audit passed.')

## 33. Modeling frame
At least one prior appearance is required so the forecast has a real historical anchor.

## 34. Target distribution
PRA is right-skewed because stars occupy the upper tail while bench players cluster near zero-to-low production.

In [ ]:
model_df = pg[
    (pg['gameDateTimeEst'] >= MODEL_START_DATE)
    & (pg['career_games_before'] >= 1)
].copy()

model_df = model_df.dropna(subset=['PRA'])
for col in CATEGORICAL_FEATURES:
    model_df[col] = model_df[col].astype('string').fillna('Unknown').astype(str)
print(f'Model rows: {len(model_df):,}')
print(f'Unique players: {model_df.personId.nunique():,}')
print(model_df['gameDateTimeEst'].min(), 'to', model_df['gameDateTimeEst'].max())

# ---- next section ----

model_df['PRA'].describe(percentiles=[.1,.25,.5,.75,.9,.95,.99]).round(2)

## 35. PRA histogram
The plot is deliberately simple and reproducible.

## 36. Relationship between recent form and next result
A strong relationship is expected, but imperfect persistence is exactly why a richer model can add value.

In [ ]:
plt.figure(figsize=(9,5))
plt.hist(model_df['PRA'].clip(upper=model_df['PRA'].quantile(.995)), bins=45)
plt.title('Distribution of Player PRA')
plt.xlabel('PRA')
plt.ylabel('Player-games')
plt.show()

# ---- next section ----

sample_corr_cols = ['PRA','lag1_PRA','PRA_mean_3','PRA_mean_5','PRA_mean_10','numMinutes_mean_10','career_pra_mean']
model_df[sample_corr_cols].corr().round(3)

## 37. Player sample-size distribution
Forecast reliability generally improves with historical sample size.

## 38. Select a showcase player
Change this single value to inspect any player available in the file.

In [ ]:
games_per_player = model_df.groupby('personId').size()
games_per_player.describe(percentiles=[.25,.5,.75,.9,.95,.99]).round(1)

# ---- next section ----

SHOWCASE_PLAYER = 'Victor Wembanyama'
showcase = model_df[model_df['playerName'].str.casefold() == SHOWCASE_PLAYER.casefold()].copy()
print(f'{SHOWCASE_PLAYER}: {len(showcase)} modeled games')
showcase[['gameDateTimeEst','playerName','PRA','PRA_mean_5','PRA_mean_10']].tail(10)

## 39. Showcase player trend
Actual PRA is compared with the pregame rolling-10 expectation.

## 40. Chronological split
The validation period is used for model selection, tuning, blend weights, and interval calibration. The test period remains untouched until final evaluation.

In [ ]:
if len(showcase):
    tail = showcase.tail(80)
    plt.figure(figsize=(11,5))
    plt.plot(tail['gameDateTimeEst'], tail['PRA'], label='Actual PRA')
    plt.plot(tail['gameDateTimeEst'], tail['PRA_mean_10'], label='Pregame rolling-10 PRA')
    plt.title(f'{SHOWCASE_PLAYER}: Recent PRA')
    plt.xlabel('Game date')
    plt.ylabel('PRA')
    plt.legend()
    plt.show()

# ---- next section ----

train_df = model_df[model_df['gameDateTimeEst'] < VALID_START_DATE].copy()
valid_df = model_df[(model_df['gameDateTimeEst'] >= VALID_START_DATE) & (model_df['gameDateTimeEst'] < TEST_START_DATE)].copy()
test_df = model_df[model_df['gameDateTimeEst'] >= TEST_START_DATE].copy()

if len(train_df) > MAX_TRAIN_ROWS:
    train_df = train_df.sort_values('gameDateTimeEst').tail(MAX_TRAIN_ROWS).copy()

split_summary = pd.DataFrame({
    'split':['train','validation','test'],
    'rows':[len(train_df),len(valid_df),len(test_df)],
    'start':[train_df.gameDateTimeEst.min(),valid_df.gameDateTimeEst.min(),test_df.gameDateTimeEst.min()],
    'end':[train_df.gameDateTimeEst.max(),valid_df.gameDateTimeEst.max(),test_df.gameDateTimeEst.max()]
})
split_summary

## 41. Split target stability
Large distribution shifts would warn that simple random cross-validation is inappropriate. The chronological setup remains the correct evaluation design either way.

## 42. Build X/y matrices
Identifier and display columns stay outside the model matrix for later diagnostics.

In [ ]:
pd.DataFrame({
    'train': train_df['PRA'].describe(),
    'validation': valid_df['PRA'].describe(),
    'test': test_df['PRA'].describe()
}).round(2)

# ---- next section ----

X_train, y_train = train_df[FEATURES], train_df['PRA']
X_valid, y_valid = valid_df[FEATURES], valid_df['PRA']
X_test, y_test = test_df[FEATURES], test_df['PRA']

## 43. Preprocessing pipeline
Numeric columns receive median imputation and standardization. Team IDs and game type receive one-hot encoding with unseen-category protection.

## 44. Evaluation helpers
MAE is the headline metric because PRA error is naturally interpreted in stat points. RMSE and R² provide complementary information, and within-3 / within-5 rates make errors more intuitive.

In [ ]:
def make_ohe():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)

numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('scaler', StandardScaler())
])
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', make_ohe())
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipe, NUMERIC_FEATURES),
    ('cat', cat_pipe, CATEGORICAL_FEATURES)
], remainder='drop')

# ---- next section ----

def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    err = np.abs(y_true - y_pred)
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': mean_squared_error(y_true, y_pred) ** 0.5,
        'R2': r2_score(y_true, y_pred),
        'Within_3': np.mean(err <= 3),
        'Within_5': np.mean(err <= 5),
    }

def metric_row(name, split, y_true, y_pred):
    return {'Model':name, 'Split':split, **regression_metrics(y_true, y_pred)}

## 45. Naive rolling-form baseline
A serious forecasting project should beat a transparent basketball baseline, not merely a constant mean.

## 46. Candidate models
The set is intentionally compact and distinct: linear shrinkage, histogram boosting, bagged random forests, and extremely randomized trees.

In [ ]:
valid_baseline = valid_df['PRA_mean_5'].fillna(train_df['PRA'].mean()).to_numpy()
test_baseline = test_df['PRA_mean_5'].fillna(train_df['PRA'].mean()).to_numpy()

baseline_metrics = pd.DataFrame([
    metric_row('Rolling-5 Baseline','Validation',y_valid,valid_baseline),
    metric_row('Rolling-5 Baseline','Test',y_test,test_baseline),
])
baseline_metrics.round(4)

# ---- next section ----

models = {
    'Ridge': Ridge(alpha=12.0),
    'HistGradientBoosting': HistGradientBoostingRegressor(
        learning_rate=0.06, max_iter=180, max_leaf_nodes=31,
        min_samples_leaf=40, l2_regularization=1.0, random_state=RANDOM_STATE
    ),
    'RandomForest': RandomForestRegressor(
        n_estimators=120, max_depth=18, min_samples_leaf=8,
        max_features=0.65, n_jobs=-1, random_state=RANDOM_STATE
    ),
    'ExtraTrees': ExtraTreesRegressor(
        n_estimators=140, max_depth=20, min_samples_leaf=6,
        max_features=0.75, n_jobs=-1, random_state=RANDOM_STATE
    ),
}

## 47. Fit candidate models
Each model owns its own preprocessing clone so there is no accidental state sharing.

## 48. Candidate-model leaderboard
Validation MAE drives model selection. Test metrics are shown for honest out-of-time assessment rather than tuning.

In [ ]:
fitted_models = {}
valid_predictions = {}
test_predictions = {}
comparison_rows = [
    metric_row('Rolling-5 Baseline','Validation',y_valid,valid_baseline),
    metric_row('Rolling-5 Baseline','Test',y_test,test_baseline)
]

for name, estimator in models.items():
    pipe = Pipeline([
        ('preprocessor', clone(preprocessor)),
        ('model', estimator)
    ])
    pipe.fit(X_train, y_train)
    fitted_models[name] = pipe
    valid_predictions[name] = pipe.predict(X_valid)
    test_predictions[name] = pipe.predict(X_test)
    comparison_rows.append(metric_row(name,'Validation',y_valid,valid_predictions[name]))
    comparison_rows.append(metric_row(name,'Test',y_test,test_predictions[name]))
    print(name, 'done')

# ---- next section ----

comparison = pd.DataFrame(comparison_rows)
comparison.sort_values(['Split','MAE']).round(4)

## 49. Validation-only ranking
This is the ranking used before any test-based conclusions.

## 50. Optional XGBoost model
If `xgboost` is installed, it is added as another strong nonlinear candidate. The notebook remains functional without it.

In [ ]:
validation_leaderboard = (
    comparison[comparison['Split']=='Validation']
    .sort_values('MAE')
    .reset_index(drop=True)
)
validation_leaderboard.round(4)

# ---- next section ----

HAS_XGB = False
try:
    from xgboost import XGBRegressor
    xgb_pipe = Pipeline([
        ('preprocessor', clone(preprocessor)),
        ('model', XGBRegressor(
            n_estimators=350, learning_rate=0.035, max_depth=7,
            min_child_weight=10, subsample=0.85, colsample_bytree=0.80,
            reg_alpha=0.15, reg_lambda=3.0, objective='reg:squarederror',
            tree_method='hist', n_jobs=-1, random_state=RANDOM_STATE
        ))
    ])
    xgb_pipe.fit(X_train, y_train)
    fitted_models['XGBoost'] = xgb_pipe
    valid_predictions['XGBoost'] = xgb_pipe.predict(X_valid)
    test_predictions['XGBoost'] = xgb_pipe.predict(X_test)
    HAS_XGB = True
    print('XGBoost fitted.')
except Exception as e:
    print('XGBoost skipped:', e)

## 51. XGBoost metrics when available

## 52. Bayesian optimization setup
Optuna's TPE sampler searches the HistGradientBoosting hyperparameter space using validation MAE. A recent training subset is used for tuning speed; the chosen configuration is then trained on the full training window.

In [ ]:
if HAS_XGB:
    xgb_metrics = pd.DataFrame([
        metric_row('XGBoost','Validation',y_valid,valid_predictions['XGBoost']),
        metric_row('XGBoost','Test',y_test,test_predictions['XGBoost'])
    ])
    comparison = pd.concat([comparison, xgb_metrics], ignore_index=True)
    display(xgb_metrics.round(4))

# ---- next section ----

HAS_OPTUNA = False
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HAS_OPTUNA = True
except Exception as e:
    print('Optuna unavailable. Set RUN_OPTUNA=False or install optuna to run this section.')
    print(e)

## 53. Pretransform the tuning matrices
Preprocessing is deterministic, so it only needs to be fit once during the hyperparameter search.

## 54. Optuna objective
The objective never sees the test labels.

In [ ]:
tune_train = train_df.sort_values('gameDateTimeEst').tail(min(TUNE_MAX_ROWS, len(train_df))).copy()
X_tune_raw = tune_train[FEATURES]
y_tune = tune_train['PRA']

tune_preprocessor = clone(preprocessor)
X_tune = tune_preprocessor.fit_transform(X_tune_raw)
X_valid_tune = tune_preprocessor.transform(X_valid)
print(X_tune.shape, X_valid_tune.shape)

# ---- next section ----

def hgb_objective(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.14, log=True),
        'max_iter': trial.suggest_int('max_iter', 140, 420),
        'max_leaf_nodes': trial.suggest_int('max_leaf_nodes', 15, 63),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 15, 100),
        'l2_regularization': trial.suggest_float('l2_regularization', 1e-3, 12.0, log=True),
        'max_bins': trial.suggest_int('max_bins', 64, 255),
        'random_state': RANDOM_STATE,
    }
    model = HistGradientBoostingRegressor(**params)
    model.fit(X_tune, y_tune)
    pred = model.predict(X_valid_tune)
    return mean_absolute_error(y_valid, pred)

## 55. Run Bayesian optimization
Twenty-five trials is enough to demonstrate a meaningful Bayesian search without turning the notebook into a long AutoML job.

## 56. Optimization history

In [ ]:
if RUN_OPTUNA and HAS_OPTUNA:
    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
    study = optuna.create_study(direction='minimize', sampler=sampler)
    study.optimize(hgb_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
    print('Best validation MAE:', study.best_value)
    print(json.dumps(study.best_params, indent=2))
else:
    study = None
    print('Optuna tuning skipped.')

# ---- next section ----

if study is not None:
    trials_df = study.trials_dataframe()
    plt.figure(figsize=(9,4))
    plt.plot(trials_df['number'], trials_df['value'], marker='o', alpha=.65)
    plt.axhline(study.best_value, linestyle='--')
    plt.title('Optuna Validation MAE by Trial')
    plt.xlabel('Trial')
    plt.ylabel('Validation MAE')
    plt.show()

## 57. Fit the tuned HistGradientBoosting model
If Optuna is unavailable, sensible fallback parameters are used so downstream cells still run.

## 58. Tuned-model metrics

In [ ]:
if study is not None:
    best_hgb_params = {**study.best_params, 'random_state':RANDOM_STATE}
else:
    best_hgb_params = dict(
        learning_rate=0.055, max_iter=300, max_leaf_nodes=31,
        min_samples_leaf=35, l2_regularization=1.5, max_bins=255,
        random_state=RANDOM_STATE
    )

tuned_hgb_pipe = Pipeline([
    ('preprocessor', clone(preprocessor)),
    ('model', HistGradientBoostingRegressor(**best_hgb_params))
])
tuned_hgb_pipe.fit(X_train, y_train)

valid_predictions['Tuned HGB'] = tuned_hgb_pipe.predict(X_valid)
test_predictions['Tuned HGB'] = tuned_hgb_pipe.predict(X_test)
fitted_models['Tuned HGB'] = tuned_hgb_pipe

# ---- next section ----

tuned_metrics = pd.DataFrame([
    metric_row('Tuned HGB','Validation',y_valid,valid_predictions['Tuned HGB']),
    metric_row('Tuned HGB','Test',y_test,test_predictions['Tuned HGB'])
])
comparison = pd.concat([comparison, tuned_metrics], ignore_index=True)
tuned_metrics.round(4)

## 59. Select models eligible for the ensemble
Only models that beat or remain close to the rolling baseline on validation are allowed into the blend.

## 60. Validation prediction diversity
Correlated errors reduce ensemble value, so this table is useful even when individual MAEs are similar.

In [ ]:
baseline_val_mae = mean_absolute_error(y_valid, valid_baseline)
model_val_mae = {
    name: mean_absolute_error(y_valid, pred)
    for name, pred in valid_predictions.items()
}
eligible_models = [name for name, mae in model_val_mae.items() if mae <= baseline_val_mae * 1.03]
if len(eligible_models) < 2:
    eligible_models = sorted(model_val_mae, key=model_val_mae.get)[:2]
eligible_models = sorted(eligible_models, key=model_val_mae.get)[:3]
print('Eligible models:', eligible_models)

# ---- next section ----

val_pred_frame = pd.DataFrame({name: valid_predictions[name] for name in eligible_models})
val_pred_frame.corr().round(3)

## 61. Nonnegative least-squares blend
Weights are fitted **only on validation predictions** and constrained nonnegative for stability and interpretability.

## 62. Ensemble predictions
The exact validation-derived weights are frozen before producing the test blend.

In [ ]:
A_valid = np.column_stack([valid_predictions[name] for name in eligible_models])
raw_weights, _ = nnls(A_valid, np.asarray(y_valid))
if raw_weights.sum() == 0:
    blend_weights = np.repeat(1/len(eligible_models), len(eligible_models))
else:
    blend_weights = raw_weights / raw_weights.sum()

blend_weight_table = pd.DataFrame({'Model':eligible_models, 'Weight':blend_weights})
blend_weight_table.sort_values('Weight', ascending=False)

# ---- next section ----

ensemble_valid = np.column_stack([valid_predictions[n] for n in eligible_models]) @ blend_weights
ensemble_test = np.column_stack([test_predictions[n] for n in eligible_models]) @ blend_weights

ensemble_metrics = pd.DataFrame([
    metric_row('NNLS Ensemble','Validation',y_valid,ensemble_valid),
    metric_row('NNLS Ensemble','Test',y_test,ensemble_test),
])
comparison = pd.concat([comparison, ensemble_metrics], ignore_index=True)
ensemble_metrics.round(4)

## 63. Final model comparison
This table is the main quantitative summary of the project.

## 64. Test-set residual distribution
Residuals reveal whether the model has systematic bias or simply irreducible game-to-game volatility.

In [ ]:
final_comparison = comparison.drop_duplicates(['Model','Split'], keep='last')
final_comparison.sort_values(['Split','MAE']).reset_index(drop=True).round(4)

# ---- next section ----

test_residuals = np.asarray(y_test) - ensemble_test
pd.Series(test_residuals).describe(percentiles=[.05,.1,.25,.5,.75,.9,.95]).round(3)

## 65. Residual plot

## 66. Error by predicted-PRA tier
The model should be evaluated separately on low-usage players, rotation players, and star-level projections.

In [ ]:
plt.figure(figsize=(9,5))
plt.scatter(ensemble_test, test_residuals, s=8, alpha=.15)
plt.axhline(0, linestyle='--')
plt.title('Ensemble Test Residuals')
plt.xlabel('Predicted PRA')
plt.ylabel('Actual - Predicted')
plt.show()

# ---- next section ----

error_df = test_df[['personId','playerName','PRA']].copy()
error_df['prediction'] = ensemble_test
error_df['abs_error'] = (error_df['PRA'] - error_df['prediction']).abs()
error_df['predicted_tier'] = pd.cut(
    error_df['prediction'], bins=[-np.inf,10,20,30,40,np.inf],
    labels=['<10','10-20','20-30','30-40','40+']
)
error_df.groupby('predicted_tier', observed=False).agg(
    games=('PRA','size'), MAE=('abs_error','mean'), actual_PRA=('PRA','mean'), predicted_PRA=('prediction','mean')
).round(2)

## 67. Error by player position
Hybrid players can belong to more than one position flag; the table therefore evaluates each broad group independently.

## 68. Error by prior role / minutes
A forecast based on stable 35-minute roles is usually easier than one for players whose minutes fluctuate sharply.

In [ ]:
position_error_rows = []
for position_col, label in [('guard','Guard'),('forward','Forward'),('center','Center')]:
    mask = test_df[position_col].fillna(0).astype(float).to_numpy() == 1
    if mask.sum():
        position_error_rows.append({
            'Position':label,
            'Games':int(mask.sum()),
            **regression_metrics(np.asarray(y_test)[mask], ensemble_test[mask])
        })
pd.DataFrame(position_error_rows).round(4)

# ---- next section ----

role_df = error_df.copy()
role_df['prior_minutes'] = test_df['numMinutes_mean_10'].to_numpy()
role_df['minutes_band'] = pd.cut(role_df['prior_minutes'], [-np.inf,10,20,30,36,np.inf])
role_df.groupby('minutes_band', observed=False).agg(games=('PRA','size'), MAE=('abs_error','mean')).round(3)

## 69. Largest overpredictions
These cases often reveal sudden role loss, injury limitation, foul trouble, or unusual game scripts.

## 70. Largest underpredictions
These cases often capture role expansion, overtime, unusually hot shooting, or breakout performances.

In [ ]:
error_df['signed_error'] = error_df['PRA'] - error_df['prediction']
error_df.sort_values('signed_error').head(20)[
    ['playerName','PRA','prediction','signed_error']
].round(2)

# ---- next section ----

error_df.sort_values('signed_error', ascending=False).head(20)[
    ['playerName','PRA','prediction','signed_error']
].round(2)

## 71. Player-level test reliability
This identifies players for whom the model is consistently accurate or consistently difficult.

## 72. Conformal-style residual interval
An 80% interval is calibrated from **validation absolute residuals**, not from test outcomes. This provides a simple distribution-free uncertainty band under approximate temporal stability.

In [ ]:
player_error = (
    error_df.groupby(['personId','playerName'])
    .agg(games=('PRA','size'), MAE=('abs_error','mean'), avg_actual=('PRA','mean'), avg_prediction=('prediction','mean'))
    .query('games >= 15')
    .sort_values('MAE')
)
player_error.head(25).round(2)

# ---- next section ----

valid_abs_resid = np.abs(np.asarray(y_valid) - ensemble_valid)
interval_radius = float(np.quantile(valid_abs_resid, INTERVAL_COVERAGE))
print(f'{int(INTERVAL_COVERAGE*100)}% interval radius: ±{interval_radius:.2f} PRA')

## 73. Check interval coverage on the untouched test period

## 74. Interval width by player volatility
A global interval is intentionally simple. This diagnostic shows why player-specific uncertainty could be a future extension.

In [ ]:
lower_test = ensemble_test - interval_radius
upper_test = ensemble_test + interval_radius
test_coverage = np.mean((np.asarray(y_test) >= lower_test) & (np.asarray(y_test) <= upper_test))
print('Observed test coverage:', round(test_coverage, 4))

# ---- next section ----

vol_diag = test_df[['playerName','PRA_std_10']].copy()
vol_diag['abs_error'] = np.abs(np.asarray(y_test) - ensemble_test)
vol_diag['volatility_quartile'] = pd.qcut(vol_diag['PRA_std_10'].rank(method='first'), 4, labels=['Q1','Q2','Q3','Q4'])
vol_diag.groupby('volatility_quartile', observed=False).agg(
    games=('abs_error','size'), mean_volatility=('PRA_std_10','mean'), MAE=('abs_error','mean')
).round(3)

## 75. Permutation importance
Permutation importance is computed on a manageable test sample and measures the loss in predictive performance when a raw feature is disrupted.

## 76. Feature-importance chart

In [ ]:
best_single_name = min(model_val_mae, key=model_val_mae.get)
best_single_model = fitted_models[best_single_name]
importance_sample = test_df.sample(min(2000, len(test_df)), random_state=RANDOM_STATE)

perm = permutation_importance(
    best_single_model,
    importance_sample[FEATURES],
    importance_sample['PRA'],
    scoring='neg_mean_absolute_error',
    n_repeats=1,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
importance_df = pd.DataFrame({
    'feature': FEATURES,
    'importance': perm.importances_mean,
    'std': perm.importances_std
}).sort_values('importance', ascending=False)
importance_df.head(25)

# ---- next section ----

top_imp = importance_df.head(20).sort_values('importance')
plt.figure(figsize=(9,7))
plt.barh(top_imp['feature'], top_imp['importance'])
plt.title(f'Permutation Importance — {best_single_name}')
plt.xlabel('Increase in MAE when permuted')
plt.show()

## 77. Recent showcase-player backtest
This is a more intuitive view of how the final ensemble tracks one player's game-to-game production.

## 78. Showcase prediction chart

In [ ]:
show_mask = test_df['playerName'].str.casefold() == SHOWCASE_PLAYER.casefold()
if show_mask.sum():
    show_bt = test_df.loc[show_mask, ['gameDateTimeEst','PRA']].copy()
    show_bt['prediction'] = ensemble_test[show_mask.to_numpy()]
    show_bt['lower80'] = show_bt['prediction'] - interval_radius
    show_bt['upper80'] = show_bt['prediction'] + interval_radius
    display(show_bt.tail(20).round(2))

# ---- next section ----

if show_mask.sum():
    plt.figure(figsize=(11,5))
    plt.plot(show_bt['gameDateTimeEst'], show_bt['PRA'], marker='o', label='Actual PRA')
    plt.plot(show_bt['gameDateTimeEst'], show_bt['prediction'], marker='o', label='Predicted PRA')
    plt.fill_between(show_bt['gameDateTimeEst'], show_bt['lower80'], show_bt['upper80'], alpha=.15, label='80% interval')
    plt.title(f'{SHOWCASE_PLAYER}: Out-of-Time PRA Forecasts')
    plt.ylabel('PRA')
    plt.xlabel('Game date')
    plt.legend()
    plt.show()

## 79. Refit production models on all labeled history
After the test evaluation is frozen, production versions can use all available labeled games. The ensemble weights remain those learned on validation.

## 80. Build helper lookups for forward forecasts
A future schedule was not supplied, so the prediction function accepts optional opponent, home/away, and date inputs. If those are omitted, neutral assumptions are used and reported explicitly.

In [ ]:
production_df = model_df.sort_values('gameDateTimeEst').tail(MAX_TRAIN_ROWS).copy()
X_prod, y_prod = production_df[FEATURES], production_df['PRA']

production_models = {}
for name in eligible_models:
    if name == 'Tuned HGB':
        model = Pipeline([
            ('preprocessor', clone(preprocessor)),
            ('model', HistGradientBoostingRegressor(**best_hgb_params))
        ])
    else:
        # Clone the already-defined candidate pipeline/model configuration.
        model = clone(fitted_models[name])
    model.fit(X_prod, y_prod)
    production_models[name] = model
    print('Refit:', name)

# ---- next section ----

TEAM_FORM_SOURCE_MAP = {f'team_{col}_mean_10': col for col in TEAM_FORM_BASE}

team_name_to_id = {}
for _, row in team_lookup.iterrows():
    full = f"{row['teamCity']} {row['teamName']}".strip().casefold()
    team_name_to_id[full] = int(row['lookupTeamId'])
    team_name_to_id[str(row['teamName']).strip().casefold()] = int(row['lookupTeamId'])
    team_name_to_id[str(row['teamAbbrev']).strip().casefold()] = int(row['lookupTeamId'])

## 81. Resolve player IDs safely
Names are convenient, but duplicate names can exist. The function warns when a name maps to multiple IDs.

## 82. Resolve opponent IDs
You can pass a full team name, nickname, abbreviation, numeric ID, or `None` for neutral opponent context.

In [ ]:
def resolve_player_id(player):
    if isinstance(player, (int, np.integer)):
        return int(player)
    matches = pg.loc[pg['playerName'].str.casefold() == str(player).casefold(), ['personId','playerName']].drop_duplicates()
    if len(matches) == 0:
        raise ValueError(f'Player not found: {player}')
    if len(matches) > 1:
        raise ValueError(f'Multiple player IDs match {player}. Pass personId instead: {matches.to_dict("records")}')
    return int(matches.iloc[0]['personId'])

# ---- next section ----

def resolve_team_id(team):
    if team is None:
        return None
    if isinstance(team, (int, np.integer)):
        return int(team)
    key = str(team).strip().casefold()
    if key not in team_name_to_id:
        raise ValueError(f'Unknown team: {team}. Examples: BOS, Knicks, San Antonio Spurs')
    return team_name_to_id[key]

## 83. Forward player-form feature builder
Unlike training rows, the synthetic next-game row should include the player's **latest completed game** inside its rolling history because that game is now in the past.

## 84. Forward team-context helper
For known teams, the latest completed team games are used. For an unknown future opponent, opponent features fall back to league medians rather than pretending a matchup is known.

In [ ]:
def build_player_form_for_next_game(player_id):
    hist = pg[pg['personId'] == player_id].sort_values(['gameDateTimeEst','gameId']).copy()
    if len(hist) < 2:
        raise ValueError('At least two historical appearances are recommended for a forward forecast.')
    last = hist.iloc[-1]
    row = {
        'playerteamId': last['playerteamId'],
        'career_games_before': len(hist),
        'career_pra_mean': hist['PRA'].mean(),
        'starter_rate_10': hist['was_starter'].tail(10).mean(),
    }
    for col in PLAYER_LAG_COLS:
        row[f'lag1_{col}'] = last[col]
    for col in ROLL_COLS:
        for w in ROLL_WINDOWS:
            vals = hist[col].tail(w)
            row[f'{col}_mean_{w}'] = vals.mean()
            row[f'{col}_std_{w}'] = vals.std(ddof=1) if len(vals) >= 2 else np.nan
    for col in USAGE_COLS:
        for w in [5,10]:
            row[f'{col}_mean_{w}'] = hist[col].tail(w).mean()
    return hist, last, row

# ---- next section ----

def latest_team_context(team_id, next_date, prefix=''):
    out = {}

    def summarize_one(tid):
        hist = ts[ts['teamId'] == tid].sort_values(['gameDateTimeEst','gameId'])
        if len(hist) == 0:
            return None
        vals = {}
        for feature_name, raw_col in TEAM_FORM_SOURCE_MAP.items():
            vals[feature_name] = hist[raw_col].tail(10).mean()
        vals['team_rest_days'] = float(np.clip((next_date - hist.iloc[-1]['gameDateTimeEst']).total_seconds()/86400, 0, 14))
        return vals

    if team_id is not None:
        vals = summarize_one(team_id)
        if vals is not None:
            return {f'{prefix}{k}': v for k, v in vals.items()}

    # Neutral context = median of current team-form summaries across franchises.
    team_summaries = [summarize_one(int(tid)) for tid in ts['teamId'].dropna().unique()]
    team_summaries = [x for x in team_summaries if x is not None]
    for feature_name in TEAM_FORM_SOURCE_MAP:
        out[f'{prefix}{feature_name}'] = float(np.nanmedian([x[feature_name] for x in team_summaries]))
    out[f'{prefix}team_rest_days'] = float(np.nanmedian([x['team_rest_days'] for x in team_summaries]))
    return out

## 85. Complete next-game row builder
The function returns both the model row and a dictionary of assumptions for transparent reporting.

## 86. Final prediction function
The headline is the ensemble PRA forecast, accompanied by component model predictions and the calibrated 80% interval.

In [ ]:
def make_next_game_features(player, opponent=None, home=None, next_game_date=None, game_type='Regular Season'):
    player_id = resolve_player_id(player)
    opponent_id = resolve_team_id(opponent)
    hist, last, row = build_player_form_for_next_game(player_id)

    if next_game_date is None:
        median_rest = hist['rest_days'].dropna().median()
        median_rest = 2.0 if pd.isna(median_rest) else float(median_rest)
        next_date = last['gameDateTimeEst'] + pd.Timedelta(days=median_rest)
        date_assumption = 'estimated from player median rest'
    else:
        next_date = pd.Timestamp(next_game_date)
        date_assumption = 'user supplied'

    if home is None:
        row['home'] = 0.5
        home_assumption = 'neutral 0.5 because home/away was not supplied'
    else:
        row['home'] = float(bool(home))
        home_assumption = 'user supplied'

    row['opponentteamId'] = opponent_id if opponent_id is not None else -1
    row['gameType'] = game_type
    row['rest_days'] = float(np.clip((next_date - last['gameDateTimeEst']).total_seconds()/86400, 0, 14))
    row['game_month'] = next_date.month
    season_year = next_date.year if next_date.month >= 7 else next_date.year - 1
    row['days_into_season'] = (next_date.normalize() - pd.Timestamp(f'{season_year}-07-01')).days

    bio = player_bio[player_bio['personId'] == player_id]
    if len(bio):
        b = bio.iloc[0]
        for c in ['heightInches','bodyWeightLbs','guard','forward','center','draftNumber']:
            row[c] = b[c]
        row['age_years'] = ((next_date - b['birthDate']).total_seconds() / (365.25*86400)) if pd.notna(b['birthDate']) else np.nan
    else:
        for c in ['heightInches','bodyWeightLbs','guard','forward','center','draftNumber','age_years']:
            row[c] = np.nan

    row.update(latest_team_context(int(last['playerteamId']) if pd.notna(last['playerteamId']) else None, next_date, prefix=''))
    row.update(latest_team_context(opponent_id, next_date, prefix='opp_'))

    out = pd.DataFrame([row])
    for c in FEATURES:
        if c not in out.columns:
            out[c] = np.nan
    out = out[FEATURES]
    for col in CATEGORICAL_FEATURES:
        out[col] = out[col].astype('string').fillna('Unknown').astype(str)

    assumptions = {
        'player_id': player_id,
        'player_name': last['playerName'],
        'last_game': last['gameDateTimeEst'],
        'forecast_game_date': next_date,
        'opponent': opponent if opponent is not None else 'neutral / unspecified',
        'opponent_context': 'known team latest form' if opponent_id is not None else 'league-median opponent form',
        'home_context': home_assumption,
        'date_context': date_assumption,
    }
    return out, assumptions

# ---- next section ----

def predict_next_pra(player, opponent=None, home=None, next_game_date=None, game_type='Regular Season'):
    X_next, assumptions = make_next_game_features(
        player=player, opponent=opponent, home=home,
        next_game_date=next_game_date, game_type=game_type
    )
    component_preds = {
        name: float(production_models[name].predict(X_next)[0])
        for name in eligible_models
    }
    pred = float(sum(component_preds[name] * weight for name, weight in zip(eligible_models, blend_weights)))
    result = {
        **assumptions,
        'predicted_PRA': pred,
        'lower_80': pred - interval_radius,
        'upper_80': pred + interval_radius,
        **{f'{name}_prediction': value for name, value in component_preds.items()}
    }
    return pd.DataFrame([result])

## 87. Example: neutral next-game forecast
This runs with only a player name. Because no future schedule is present in the supplied files, the model explicitly uses neutral home/away and league-median opponent context.

## 88. Example: matchup-aware forecast
Replace the opponent, home flag, and date with the actual upcoming matchup when available.

In [ ]:
predict_next_pra(SHOWCASE_PLAYER).round(2)

# ---- next section ----

# Example only — edit these values for a real upcoming matchup.
MATCHUP_EXAMPLE = predict_next_pra(
    SHOWCASE_PLAYER,
    opponent='NYK',
    home=True,
    next_game_date=None
)
MATCHUP_EXAMPLE.round(2)

## 89. User input cell
This is the main cell to edit after the notebook has been run once from top to bottom.

## 90. Produce a latest-player prediction board
This board forecasts players whose most recent appearance is close to the dataset's latest game date. With no schedule, these are neutral-context forecasts.

In [ ]:
PLAYER_TO_PREDICT = 'Victor Wembanyama'
NEXT_OPPONENT = None      # e.g. 'BOS', 'Knicks', or None
NEXT_GAME_IS_HOME = None  # True / False / None
NEXT_GAME_DATE = None     # e.g. '2026-10-22' or None

NEXT_PRA_FORECAST = predict_next_pra(
    PLAYER_TO_PREDICT,
    opponent=NEXT_OPPONENT,
    home=NEXT_GAME_IS_HOME,
    next_game_date=NEXT_GAME_DATE
)
NEXT_PRA_FORECAST.round(2)

# ---- next section ----

latest_date = pg['gameDateTimeEst'].max()
recent_cutoff = latest_date - pd.Timedelta(days=45)
recent_players = (
    pg.groupby(['personId','playerName'], as_index=False)['gameDateTimeEst'].max()
      .query('gameDateTimeEst >= @recent_cutoff')
      .sort_values('gameDateTimeEst', ascending=False)
)
print('Players eligible for recent board:', len(recent_players))

## 91. Generate the neutral-context board
To keep interactive runs reasonable, the default board is capped at 150 recent players; increase the cap if desired.

## 92. Export prediction board
The CSV contains the forecast, uncertainty interval, assumptions, and component-model predictions.

In [ ]:
BOARD_LIMIT = 150
board_rows = []
for _, r in recent_players.head(BOARD_LIMIT).iterrows():
    try:
        pred = predict_next_pra(int(r['personId']))
        board_rows.append(pred.iloc[0].to_dict())
    except Exception:
        pass

prediction_board = pd.DataFrame(board_rows)
if len(prediction_board):
    prediction_board = prediction_board.sort_values('predicted_PRA', ascending=False).reset_index(drop=True)
prediction_board.head(30).round(2)

# ---- next section ----

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
board_path = OUTPUT_DIR / 'next_game_pra_predictions.csv'
prediction_board.to_csv(board_path, index=False)
print(board_path.resolve())

## 93. Export model comparison and blend weights
These artifacts make the notebook's model-selection logic auditable outside Jupyter.

## 94. Save a compact run summary
This JSON captures the main evaluation numbers, selected ensemble members, weights, and uncertainty radius.

In [ ]:
comparison_path = OUTPUT_DIR / 'pra_model_comparison.csv'
weights_path = OUTPUT_DIR / 'pra_ensemble_weights.csv'
final_comparison.to_csv(comparison_path, index=False)
blend_weight_table.to_csv(weights_path, index=False)
print(comparison_path.resolve())
print(weights_path.resolve())

# ---- next section ----

summary = {
    'target': 'next-game PRA',
    'train_start': str(train_df.gameDateTimeEst.min()),
    'validation_start': str(VALID_START_DATE),
    'test_start': str(TEST_START_DATE),
    'test_end': str(test_df.gameDateTimeEst.max()),
    'ensemble_models': eligible_models,
    'ensemble_weights': {m: float(w) for m, w in zip(eligible_models, blend_weights)},
    'ensemble_test_metrics': regression_metrics(y_test, ensemble_test),
    'interval_coverage_target': INTERVAL_COVERAGE,
    'interval_radius': interval_radius,
    'observed_test_interval_coverage': float(test_coverage),
}
summary_path = OUTPUT_DIR / 'pra_run_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

## 95. What the model is learning
The strongest signals should generally be interpretable basketball quantities: recent PRA, recent minutes, scoring/rebounding/assist form, offensive usage, role stability, rest, team environment, opponent defensive form, age, and position.

The exact ordering should be read from the permutation-importance output rather than assumed in advance.

## 96. Limitations
- **No future schedule is included**, so true forward matchup inputs must be supplied manually when known.
- Injury designations, projected lineups, betting totals, travel distance, and live availability news are absent.
- The target is conditional on an appearance in the player-game data; explicit DNP probability is not modeled.
- Historical NBA eras differ, which is why the production model emphasizes modern seasons.
- A global 80% residual interval is simple and transparent but does not fully adapt to player-specific volatility.

In [ ]:
# Sections 95-96 are explanatory; no executable code in this block.

## 97. Why this is a stronger sports-analytics prediction setup
This project demonstrates more than fitting a regressor. It separates **what is known before the game** from what happens during the game, uses chronological validation, integrates player and team context, benchmarks against a basketball-specific rolling baseline, applies Bayesian optimization, blends models using out-of-time predictions, quantifies uncertainty, and exposes a reusable player-level forecasting function.

## 98. Portfolio summary
**NBA Next-Game PRA Forecasting System**

Built an end-to-end, leakage-safe machine-learning pipeline to predict NBA player Points + Rebounds + Assists for the next game. Engineered lagged player form, workload, role, team, opponent, rest, and player-profile features from four historical datasets; evaluated models with strict chronological holdouts; tuned gradient boosting with Bayesian optimization; combined diverse regressors through validation-based nonnegative ensembling; calibrated forecast intervals; analyzed errors and feature importance; and created a reusable matchup-aware prediction API plus exportable prediction board.

In [ ]:
# Sections 97-98 are explanatory; no executable code in this block.